# PLE dataset builder (Kaggle-side, CPU, nothing stages locally)
Pushes 33 pinned FP8 PLE shards (~48.7 GiB) + manifest.json in 11 additive versions (3 shards each).
Each shard is size-verified against the pinned HF revision before its version push.


In [ ]:
# Cell 0 - setup: environment credentials and kaggle CLI
import os, subprocess, sys, json, hashlib, shutil, time
from pathlib import Path
HF = os.environ.get('HF_TOKEN')
KG = os.environ.get('KAGGLE_API_TOKEN')
if not HF or not KG:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        HF = HF or user_secrets.get_secret('HF_TOKEN')
        KG = KG or user_secrets.get_secret('KG_TOKEN')
    except Exception:
        pass
assert HF and KG, 'no Kaggle/HF credentials'
os.environ['KAGGLE_API_TOKEN'] = KG
os.environ['HF_TOKEN'] = HF
%pip install -q kaggle==2.2.4
WORK=Path('/kaggle/working/ple-build'); STAGE=WORK/'stage'; META=WORK/'meta'
STAGE.mkdir(parents=True, exist_ok=True); META.mkdir(parents=True, exist_ok=True)
print('kaggle', __import__('kaggle').__version__, '| staging under', WORK)


In [ ]:
# Cell 1 - payload: exact shard list + manifest (authored offline from pinned index)
PLE_SRC='Qwen/Qwen3.8-Flash-Next-FP8'
PLE_REV='236dfdf285828023ca3bcd3f37366c58a3469b13'
DS_SLUG='ninnix/qwen38-flashnext-ple-fp8'
SHARDS=["model-00005-of-00131.safetensors", "model-00006-of-00131.safetensors", "model-00007-of-00131.safetensors", "model-00008-of-00131.safetensors", "model-00009-of-00131.safetensors", "model-00010-of-00131.safetensors", "model-00011-of-00131.safetensors", "model-00012-of-00131.safetensors", "model-00013-of-00131.safetensors", "model-00014-of-00131.safetensors", "model-00015-of-00131.safetensors", "model-00016-of-00131.safetensors", "model-00017-of-00131.safetensors", "model-00018-of-00131.safetensors", "model-00019-of-00131.safetensors", "model-00020-of-00131.safetensors", "model-00021-of-00131.safetensors", "model-00022-of-00131.safetensors", "model-00023-of-00131.safetensors", "model-00024-of-00131.safetensors", "model-00025-of-00131.safetensors", "model-00026-of-00131.safetensors", "model-00027-of-00131.safetensors", "model-00028-of-00131.safetensors", "model-00029-of-00131.safetensors", "model-00030-of-00131.safetensors", "model-00031-of-00131.safetensors", "model-00032-of-00131.safetensors", "model-00033-of-00131.safetensors", "model-00034-of-00131.safetensors", "model-00035-of-00131.safetensors", "model-00036-of-00131.safetensors", "model-00037-of-00131.safetensors"]
MANIFEST={"format": "qwen-flashnext-ple-fp8", "version": 1, "ple_revision": "236dfdf285828023ca3bcd3f37366c58a3469b13", "source_model": "Qwen/Qwen3.8-Flash-Next-FP8", "rows_per_part": 2500012, "row_dim": 160, "parts": {"0": "model-00005-of-00131.safetensors", "1": "model-00005-of-00131.safetensors", "2": "model-00006-of-00131.safetensors", "3": "model-00006-of-00131.safetensors", "4": "model-00006-of-00131.safetensors", "5": "model-00006-of-00131.safetensors", "6": "model-00007-of-00131.safetensors", "7": "model-00007-of-00131.safetensors", "8": "model-00007-of-00131.safetensors", "9": "model-00007-of-00131.safetensors", "10": "model-00008-of-00131.safetensors", "11": "model-00008-of-00131.safetensors", "12": "model-00008-of-00131.safetensors", "13": "model-00008-of-00131.safetensors", "14": "model-00009-of-00131.safetensors", "15": "model-00009-of-00131.safetensors", "16": "model-00009-of-00131.safetensors", "17": "model-00009-of-00131.safetensors", "18": "model-00010-of-00131.safetensors", "19": "model-00010-of-00131.safetensors", "20": "model-00010-of-00131.safetensors", "21": "model-00010-of-00131.safetensors", "22": "model-00011-of-00131.safetensors", "23": "model-00011-of-00131.safetensors", "24": "model-00011-of-00131.safetensors", "25": "model-00011-of-00131.safetensors", "26": "model-00012-of-00131.safetensors", "27": "model-00012-of-00131.safetensors", "28": "model-00012-of-00131.safetensors", "29": "model-00012-of-00131.safetensors", "30": "model-00013-of-00131.safetensors", "31": "model-00013-of-00131.safetensors", "32": "model-00013-of-00131.safetensors", "33": "model-00013-of-00131.safetensors", "34": "model-00014-of-00131.safetensors", "35": "model-00014-of-00131.safetensors", "36": "model-00014-of-00131.safetensors", "37": "model-00014-of-00131.safetensors", "38": "model-00015-of-00131.safetensors", "39": "model-00015-of-00131.safetensors", "40": "model-00015-of-00131.safetensors", "41": "model-00015-of-00131.safetensors", "42": "model-00016-of-00131.safetensors", "43": "model-00016-of-00131.safetensors", "44": "model-00016-of-00131.safetensors", "45": "model-00016-of-00131.safetensors", "46": "model-00017-of-00131.safetensors", "47": "model-00017-of-00131.safetensors", "48": "model-00017-of-00131.safetensors", "49": "model-00017-of-00131.safetensors", "50": "model-00018-of-00131.safetensors", "51": "model-00018-of-00131.safetensors", "52": "model-00018-of-00131.safetensors", "53": "model-00018-of-00131.safetensors", "54": "model-00019-of-00131.safetensors", "55": "model-00019-of-00131.safetensors", "56": "model-00019-of-00131.safetensors", "57": "model-00019-of-00131.safetensors", "58": "model-00020-of-00131.safetensors", "59": "model-00020-of-00131.safetensors", "60": "model-00020-of-00131.safetensors", "61": "model-00020-of-00131.safetensors", "62": "model-00021-of-00131.safetensors", "63": "model-00021-of-00131.safetensors", "64": "model-00021-of-00131.safetensors", "65": "model-00021-of-00131.safetensors", "66": "model-00022-of-00131.safetensors", "67": "model-00022-of-00131.safetensors", "68": "model-00022-of-00131.safetensors", "69": "model-00022-of-00131.safetensors", "70": "model-00023-of-00131.safetensors", "71": "model-00023-of-00131.safetensors", "72": "model-00023-of-00131.safetensors", "73": "model-00023-of-00131.safetensors", "74": "model-00024-of-00131.safetensors", "75": "model-00024-of-00131.safetensors", "76": "model-00024-of-00131.safetensors", "77": "model-00024-of-00131.safetensors", "78": "model-00025-of-00131.safetensors", "79": "model-00025-of-00131.safetensors", "80": "model-00025-of-00131.safetensors", "81": "model-00025-of-00131.safetensors", "82": "model-00026-of-00131.safetensors", "83": "model-00026-of-00131.safetensors", "84": "model-00026-of-00131.safetensors", "85": "model-00026-of-00131.safetensors", "86": "model-00027-of-00131.safetensors", "87": "model-00027-of-00131.safetensors", "88": "model-00027-of-00131.safetensors", "89": "model-00027-of-00131.safetensors", "90": "model-00028-of-00131.safetensors", "91": "model-00028-of-00131.safetensors", "92": "model-00028-of-00131.safetensors", "93": "model-00028-of-00131.safetensors", "94": "model-00029-of-00131.safetensors", "95": "model-00029-of-00131.safetensors", "96": "model-00029-of-00131.safetensors", "97": "model-00029-of-00131.safetensors", "98": "model-00030-of-00131.safetensors", "99": "model-00030-of-00131.safetensors", "100": "model-00030-of-00131.safetensors", "101": "model-00030-of-00131.safetensors", "102": "model-00031-of-00131.safetensors", "103": "model-00031-of-00131.safetensors", "104": "model-00031-of-00131.safetensors", "105": "model-00031-of-00131.safetensors", "106": "model-00032-of-00131.safetensors", "107": "model-00032-of-00131.safetensors", "108": "model-00032-of-00131.safetensors", "109": "model-00032-of-00131.safetensors", "110": "model-00033-of-00131.safetensors", "111": "model-00033-of-00131.safetensors", "112": "model-00033-of-00131.safetensors", "113": "model-00033-of-00131.safetensors", "114": "model-00034-of-00131.safetensors", "115": "model-00034-of-00131.safetensors", "116": "model-00034-of-00131.safetensors", "117": "model-00034-of-00131.safetensors", "118": "model-00035-of-00131.safetensors", "119": "model-00035-of-00131.safetensors", "120": "model-00035-of-00131.safetensors", "121": "model-00035-of-00131.safetensors", "122": "model-00036-of-00131.safetensors", "123": "model-00036-of-00131.safetensors", "124": "model-00036-of-00131.safetensors", "125": "model-00036-of-00131.safetensors", "126": "model-00037-of-00131.safetensors", "127": "model-00037-of-00131.safetensors"}}
assert len(SHARDS)==33 and len(MANIFEST['parts'])==128
assert sorted(set(MANIFEST['parts'].values()))==SHARDS
(META/'manifest.json').write_text(json.dumps(MANIFEST, indent=2)+'\n')
(META/'dataset-metadata.json').write_text(json.dumps({'id': DS_SLUG, 'title': 'Qwen3.8-Flash-Next FP8 PLE shards (pinned)', 'licenses': [{'name': 'unknown'}]}, indent=2)+'\n')
print('payload ready:', len(SHARDS), 'shards + manifest.json')


In [ ]:
# Cell 2 - plan: 11 independent datasets (replace-semantics-proof, no staging limits)
print('\n'.join('ninnix/qwen38-ple-p%02d <- %s' % (i, ', '.join(SHARDS[3*i:3*i+3])) for i in range(11)) + '\nmanifest.json travels in every dataset')


In [ ]:
# Cell 3 - per batch of 3: skip if dataset exists, else download -> size+sha verify -> CREATE one dataset.
# Independent single-version datasets: no staging limits, no re-uploads, no processing waits.
from huggingface_hub import HfApi, hf_hub_download
api=HfApi(token=HF)
_info=api.model_info(PLE_SRC, revision=PLE_REV, files_metadata=True)
_meta={s.rfilename: (s.size, (getattr(s.lfs, 'oid', None) or getattr(s.lfs, 'sha256', None) or getattr(s.lfs, 'sha', None) if s.lfs else None)) for s in _info.siblings if s.rfilename in SHARDS}
assert len(_meta)==33 and all(v[0] for v in _meta.values()), 'incomplete file metadata'
made=[]
for bi in range(0, len(SHARDS), 3):
    tag='p%02d' % (bi//3); batch=SHARDS[bi:bi+3]; slug='ninnix/qwen38-ple-'+tag
    _r=subprocess.run(['kaggle','datasets','files',slug],capture_output=True,text=True,timeout=300)
    _listed={l.split()[0].split('/')[-1] for l in _r.stdout.splitlines() if '.safetensors' in l or 'manifest.json' in l}
    if set(batch)|{'manifest.json'}<=_listed:
        print('exists, skip:', slug, flush=True); made.append(slug); continue
    B=STAGE/('batch-'+tag); B.mkdir(parents=True, exist_ok=True)
    shutil.copy(META/'manifest.json', B/'manifest.json')
    (B/'dataset-metadata.json').write_text(json.dumps({'id': slug, 'title': 'Qwen3.8-Flash-Next PLE '+tag+' (pinned)', 'licenses': [{'name': 'unknown'}]}, indent=2)+'\n')
    got=0
    for f in batch:
        p=hf_hub_download(PLE_SRC, f, revision=PLE_REV, token=HF, local_dir=str(B))
        assert Path(p).name==f, Path(p)
        sz=Path(p).stat().st_size
        assert sz==_meta[f][0], (f, sz, _meta[f][0])
        _h=hashlib.sha256(); _fh=open(p,'rb')
        [_h.update(c) for c in iter(lambda: _fh.read(64*1024*1024), b'')]; _fh.close()
        assert _meta[f][1] is None or _h.hexdigest()==_meta[f][1], (f, 'sha256 mismatch')
        got+=sz
    print('batch %s downloaded %.2f GiB' % (tag, got/1024**3), flush=True)
    _vr=subprocess.run(['kaggle','datasets','create','-p',str(B)],capture_output=True,text=True,timeout=1800)
    print((_vr.stdout or '')[-800:], flush=True)
    assert _vr.returncode==0, ((_vr.stdout or '')+(_vr.stderr or ''))[-2000:]
    shutil.rmtree(B)
    made.append(slug)
    print('dataset %s submitted (%d/11)' % (slug, len(made)), flush=True)
print('ALL 11 DATASETS SUBMITTED')


In [ ]:
# Cell 4 - finalize: poll each dataset until its exact 4-file set is ready (20 min cap each)
import time
allfiles=set()
for i in range(11):
    slug='ninnix/qwen38-ple-p%02d' % i; want=set(SHARDS[3*i:3*i+3])|{'manifest.json'}
    _t0=time.time()
    while True:
        _r=subprocess.run(['kaggle','datasets','files',slug],capture_output=True,text=True,timeout=300)
        _listed={l.split()[0].split('/')[-1] for l in _r.stdout.splitlines() if '.safetensors' in l or 'manifest.json' in l}
        if _listed==want: break
        assert time.time()-_t0 < 1200, (slug, 'not ready', sorted(_listed))
        time.sleep(60)
    allfiles|=want
    print(slug, 'ready', flush=True)
assert allfiles==set(SHARDS)|{'manifest.json'}
print('PLE DATASETS COMPLETE: 11 x (3 shards + manifest). Attach all via dataset_sources, then run the 10K real-PLE smoke')
